# Problem statement
A social service agency receives client and assistance files. The data is messy and contains sensitive personal information (NRIC, name). Client details such as **address and household income change over time** and these changes matter when we analyse who received assistance.
## Business question
*How much assistance goes to different household income groups, across regions and schemes and how does this change over time?*
## Architecture
`Raw CSV files` -> **Bronze** (raw, as strings) -> **Silver** (cleaned, validated, masked) -> **Gold** (star schema)
| Layer | Purpose |
|---|---|
| Bronze | Keep the raw data exactly as received. Add `ingest_ts` and `source_file`. Log every load in a control table. |
| Silver | Clean, check types, validate, remove duplicates, mask PII. Bad rows go to quarantine tables. |
| Gold | Star schema: `dim_client` (SCD2), `dim_scheme`, `dim_date`, `fact_assistance`. |

## The data
The data is **synthetic** (made with Python Faker), so no real personal data is used. I added data quality problems on purpose, so I can show how the pipeline handles each one.

| File | What it is | Used for |
|---|---|---|
| `schemes.csv` | 8 assistance schemes and their categories | Reference data |
| `clients_day1.csv` | First client snapshot | Initial load |
| `clients_day2.csv` | Second snapshot: some clients moved, some are new | SCD2 and remediation |
| `assistance.csv` | Assistance cases (one row per application) | Fact table |

In [0]:
RAW = "/Volumes/workspace/bronze/raw_files"
def read_raw(file_name):
    """Read a CSV with every column as a string, exactly like the source."""
    return (spark.read
            .option("header", True)
            .option("inferSchema", False)
            .option("multiLine", True)
            .option("escape", '"')
            .csv(f"{RAW}/{file_name}"))
 
 
schemes = read_raw("schemes.csv")
clients1 = read_raw("clients_day1.csv")
clients2 = read_raw("clients_day2.csv")
assistance = read_raw("assistance.csv")
 
schemes.createOrReplaceTempView("raw_schemes")
clients1.createOrReplaceTempView("raw_clients_day1")
clients2.createOrReplaceTempView("raw_clients_day2")
assistance.createOrReplaceTempView("raw_assistance")

In [0]:
display(spark.sql("""
    SELECT 'schemes.csv' AS file, count(*) AS `rows` FROM raw_schemes
    UNION ALL SELECT 'clients_day1.csv', count(*) FROM raw_clients_day1
    UNION ALL SELECT 'clients_day2.csv', count(*) FROM raw_clients_day2
    UNION ALL SELECT 'assistance.csv', count(*) FROM raw_assistance
"""))

file,rows
schemes.csv,8
clients_day1.csv,1005
clients_day2.csv,1020
assistance.csv,2505


In [0]:
display(schemes)

scheme_id,scheme_name,category
SCH01,ComCare Short-to-Medium Term Assistance,Financial Assistance
SCH02,ComCare Long-Term Assistance,Financial Assistance
SCH03,Childcare Subsidy Top-up,Family Support
SCH04,Student Care Fee Assistance,Family Support
SCH05,Utilities Support,Living Expenses
SCH06,Transport Support,Living Expenses
SCH07,Home Ownership Support,Housing
SCH08,Employment Assistance,Employment


In [0]:
display(clients1.limit(5))

client_id,nric,full_name,address,region,household_income,household_size,updated_date
1,S8637940C,William Jennings,"Flat 32, Sian Streets",North,2787.0,3,2026-10-01
2,S3511615F,Samuel Wright,1 Donald Parkways,North-East,2166.0,7,2026-10-01
3,S7816184J,James Moore,"Flat 79, Barker Avenue",North-East,2962.0,5,2026-10-01
4,S1034131G,Mr Damien Oliver,"Flat 35, Colin Light",West,3714.0,6,2026-10-01
5,T2553419J,Eleanor Baker,"Flat 7, Gemma Common",East,4375.0,6,2026-10-01


In [0]:
display(assistance.limit(5))

case_id,client_id,scheme_id,application_date,assistance_amount,status
1,898,SCH03,2026-07-16,1481.63,Disbursed
2,910,SCH06,2026-03-26,1034.65,Approved
3,1018,SCH05,2025-10-26,655.08,Pending
4,533,SCH06,2026-05-01,109.59,Pending
5,815,SCH08,2026-09-18,228.29,Disbursed


## Data profiling: what is wrong with the raw data?
Counts are rows in the raw file, so duplicate rows are included.

In [0]:
# Clients (Day 1)
display(spark.sql(r"""
    SELECT count(*)                                   AS total_rows,
           count(DISTINCT client_id)                  AS distinct_clients,
           count(*) - count(DISTINCT client_id)       AS duplicate_rows,
           sum(CASE WHEN nric != upper(trim(nric)) THEN 1 ELSE 0 END)                           AS nric_wrong_case_or_spaces,
           sum(CASE WHEN NOT upper(trim(nric)) RLIKE '^[ST][0-9]{7}[A-Z]$' THEN 1 ELSE 0 END)   AS nric_invalid,
           sum(CASE WHEN household_income IS NULL THEN 1 ELSE 0 END)                            AS income_missing,
           sum(CASE WHEN try_cast(household_income AS DOUBLE) < 0 THEN 1 ELSE 0 END)            AS income_negative,
           sum(CASE WHEN full_name != trim(full_name) THEN 1 ELSE 0 END)                        AS name_extra_spaces
    FROM raw_clients_day1
"""))

total_rows,distinct_clients,duplicate_rows,nric_wrong_case_or_spaces,nric_invalid,income_missing,income_negative,name_extra_spaces
1005,1000,5,6,6,6,6,6


In [0]:
# Assistance
display(spark.sql(r"""
    SELECT count(*) AS total_rows,
           count(DISTINCT case_id) AS distinct_cases,
           count(*) - count(DISTINCT case_id) AS duplicate_rows,
           sum(CASE WHEN assistance_amount IS NULL THEN 1 ELSE 0 END) AS amount_missing,
           sum(CASE WHEN try_cast(assistance_amount AS DOUBLE) < 0 THEN 1 ELSE 0 END)  AS amount_negative,
           sum(CASE WHEN scheme_id NOT IN (SELECT scheme_id FROM raw_schemes) THEN 1 ELSE 0 END) AS unknown_scheme,
           sum(CASE WHEN client_id NOT IN (SELECT client_id FROM raw_clients_day1) THEN 1 ELSE 0 END) AS client_not_in_day1,
           sum(CASE WHEN try_cast(application_date AS DATE) > current_date() THEN 1 ELSE 0 END) AS future_date
    FROM raw_assistance
"""))

total_rows,distinct_cases,duplicate_rows,amount_missing,amount_negative,unknown_scheme,client_not_in_day1,future_date
2505,2500,5,10,10,10,58,10


In [0]:
# Day 2 compared with Day 1: what changed?
display(spark.sql("""
    SELECT
      (SELECT count(DISTINCT d2.client_id)
         FROM raw_clients_day2 d2
         JOIN (SELECT DISTINCT client_id, address FROM raw_clients_day1) d1
           ON d1.client_id = d2.client_id AND trim(d1.address) != trim(d2.address)) AS clients_changed_address,
      (SELECT count(*) FROM raw_clients_day2
        WHERE client_id NOT IN (SELECT client_id FROM raw_clients_day1)) AS new_clients,
      (SELECT count(*) FROM raw_clients_day2
        WHERE NOT upper(trim(nric)) RLIKE '^[ST][0-9]{7}[A-Z]$') AS invalid_nric_rows
"""))

clients_changed_address,new_clients,invalid_nric_rows
50,20,10
